In [0]:
# Create widgets
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")


# Retrieve widget values into variables
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print(f"table_name        : {table_name}")
print(f"landing_timestamp : {landing_timestamp}")
print(f"merge_keys        : {merge_keys}")
print(f"load_type         : {load_type}")

In [0]:
bronze_df = spark.read.table(f"dbx_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

record_count = bronze_df.count()
print(f"Records read from bronze : {record_count}")
print(f"Columns ({len(bronze_df.columns)}) : {', '.join(bronze_df.columns)}")
print("")
bronze_df.printSchema()

In [0]:
from pyspark.sql import functions as F
updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("updated_timestamp", F.current_timestamp())

print("Sample data with audit timestamps (5 rows):")
display(updated_df.limit(5))

In [0]:
from delta.tables import DeltaTable

silver_table = f"dbx_caresync.silver.{table_name}"
source_count = updated_df.count()

print(f"{'='*55}")
print(f"  Target table : {silver_table}")
print(f"  Load type    : {load_type}")
print(f"  Source rows  : {source_count}")
print(f"{'='*55}")

if load_type == "FULL":
    print("Action : overwrite target table")
    updated_df.write.mode("overwrite").saveAsTable(silver_table)
    print(f"FULL load complete -> {silver_table}")

elif load_type == "APPEND":
    print("Action : append rows to target table")
    updated_df.write.mode("append").saveAsTable(silver_table)
    after_count = spark.read.table(silver_table).count()
    print(f"APPEND load complete -> {silver_table}")
    print(f"Total rows in target after append : {after_count}")

elif load_type == "MERGE":
    if not merge_keys:
        raise ValueError("merge_keys must be provided for MERGE load type.")

    keys = [k.strip() for k in merge_keys.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

    if spark.catalog.tableExists(silver_table):
        before_count = spark.read.table(silver_table).count()
        print(f"Merge keys      : {merge_keys}")
        print(f"Merge condition : {merge_condition}")
        print(f"Rows in target before merge : {before_count}")
        print("Action : merge source into target")

        # Exclude insert_timestamp from updates — it must only be set on first insert
        update_cols = {c: f"source.{c}" for c in updated_df.columns if c != "insert_timestamp"}

        DeltaTable.forName(spark, silver_table) \
            .alias("target") \
            .merge(updated_df.alias("source"), merge_condition) \
            .whenMatchedUpdate(set=update_cols) \
            .whenNotMatchedInsertAll() \
            .execute()

        after_count = spark.read.table(silver_table).count()
        print(f"MERGE complete -> {silver_table} (keys: {merge_keys})")
        print(f"Rows in target after merge  : {after_count}")
        print(f"Net change                  : {after_count - before_count:+d} rows")
    else:
        # First load - table does not exist yet; do initial full write
        print("Target table does not exist yet — running initial full write")
        updated_df.write.mode("overwrite").saveAsTable(silver_table)
        print(f"MERGE (initial load, table did not exist) -> {silver_table}")

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected FULL, APPEND, or MERGE.")

In [0]:
records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))